# Dados não mentem... ou mentem?
## Investigação de qualidade com PySpark · TDC · Agnes Ruescas


In [0]:
# C01 — Ambiente
from pyspark.sql import functions as F, Window
from decimal import Decimal

META = Decimal("1000.00")  # Premissa fictícia do caso
print("Spark:", spark.version)

def reais(valor):
    return "R$ " + f"{valor:,.2f}".replace(",", "_").replace(".", ",").replace("_", ".")


Spark: 4.2.0


## Preparação dos dados
Contrato: um pedido na versão mais recente, cliente e pedido preenchidos, valor decimal positivo e status conhecido. Somente status `pago` entra na soma.

Neste lote, `versao` indica a atualização e `evento_id` desempata. Em produção, a origem precisa garantir a semântica desses campos. A entrada contém eventos.

In [0]:
# C02 — Dados do caso
dados = [('001', 'P01', 'C01', '100.00', 'pago', 1),
 ('002', 'P02', 'C02', '200.00', 'pago', 1),
 ('002', 'P02', 'C02', '200.00', 'pago', 1),
 ('003', 'P03', 'C03', '120.00', 'pago', 1),
 ('004', 'P03', 'C03', '150.00', 'pago', 2),
 ('005', 'P04', 'C04', None, 'pago', 1),
 ('006', 'P05', 'C05', '-50.00', 'pago', 1),
 ('007', 'P06', 'C06', 'abc', 'pago', 1),
 ('008', 'P07', 'C07', '80.00', 'cancelado', 1),
 ('009', 'P08', None, '300.00', 'pago', 1),
 ('010', 'P09', 'C09', '250.00', 'pago', 1),
 ('011', 'P10', 'C10', '100.00', 'pago', 1)]
colunas = "evento_id string, pedido_id string, cliente_id string, valor_raw string, status string, versao int"
raw = spark.createDataFrame(dados, colunas)
print("Lote sintético carregado.")


Lote sintético carregado.


## 1. Reproduzindo o relatório
A soma inicial sustenta o gráfico da reunião?

In [0]:
# C03 — Relatório inicial
base = raw.withColumn("valor", F.expr("try_cast(valor_raw as decimal(12,2))"))
inicial = base.filter(F.col("status") == "pago").agg(F.sum("valor").alias("total")).first()["total"]
print("Apurado:", reais(inicial))
print("Meta:", reais(META))
print(f"Variação em relação à meta: {(inicial/META-1)*100:+.0f}%")


Apurado: R$ 1.370,00
Meta: R$ 1.000,00
Variação em relação à meta: +37%


## 2. O que entrou nessa soma?
Quantas linhas têm valor? Quais pedidos aparecem mais de uma vez?

In [0]:
# C04 — Inspeção das linhas
display(base.orderBy("pedido_id", "versao"))
base.agg(F.count("*").alias("linhas"), F.count("valor").alias("valores_convertidos")).show()


evento_id,pedido_id,cliente_id,valor_raw,status,versao,valor
001,P01,C01,100.00,pago,1,100.00
002,P02,C02,200.00,pago,1,200.00
002,P02,C02,200.00,pago,1,200.00
003,P03,C03,120.00,pago,1,120.00
004,P03,C03,150.00,pago,2,150.00
005,P04,C04,null,pago,1,null
006,P05,C05,-50.00,pago,1,-50.00
007,P06,C06,abc,pago,1,null
008,P07,C07,80.00,cancelado,1,80.00
009,P08,null,300.00,pago,1,300.00


+------+-------------------+
|linhas|valores_convertidos|
+------+-------------------+
|    12|                 10|
+------+-------------------+



In [0]:
# C05 — Perfil e chaves
perfil = base.agg(
    F.count("*").alias("linhas"),
    F.sum(F.col("valor_raw").isNull().cast("int")).alias("valor_ausente"),
    F.sum((F.col("valor_raw").isNotNull() & F.col("valor").isNull()).cast("int")).alias("falha_conversao"),
    F.sum((F.col("valor") <= 0).cast("int")).alias("valor_nao_positivo"),
    F.sum(F.col("cliente_id").isNull().cast("int")).alias("cliente_ausente")
)
perfil.show(truncate=False)
base.groupBy("pedido_id").count().filter("count > 1").orderBy("pedido_id").show()


+------+-------------+---------------+------------------+---------------+
|linhas|valor_ausente|falha_conversao|valor_nao_positivo|cliente_ausente|
+------+-------------+---------------+------------------+---------------+
|12    |1            |1              |1                 |1              |
+------+-------------+---------------+------------------+---------------+

+---------+-----+
|pedido_id|count|
+---------+-----+
|      P02|    2|
|      P03|    2|
+---------+-----+



## 3. Repetição ou atualização?
Compare P02 e P03. Remova eventos idênticos e depois selecione a versão atual de cada pedido.

In [0]:
# C06 — Versão atual
eventos = base.dropDuplicates()  # todas as colunas
janela = Window.partitionBy("pedido_id").orderBy(
    F.col("versao").desc(), F.col("evento_id").desc()
)
atuais = (eventos.withColumn("rn", F.row_number().over(janela))
          .filter("rn = 1").drop("rn"))
display(atuais.orderBy("pedido_id"))


evento_id,pedido_id,cliente_id,valor_raw,status,versao,valor
001,P01,C01,100.00,pago,1,100.00
002,P02,C02,200.00,pago,1,200.00
004,P03,C03,150.00,pago,2,150.00
005,P04,C04,null,pago,1,null
006,P05,C05,-50.00,pago,1,-50.00
007,P06,C06,abc,pago,1,null
008,P07,C07,80.00,cancelado,1,80.00
009,P08,null,300.00,pago,1,300.00
010,P09,C09,250.00,pago,1,250.00
011,P10,C10,100.00,pago,1,100.00


## 4. Quais pedidos respeitam o contrato?
Toda regra deve retornar verdadeiro ou falso, inclusive quando o dado está ausente. Preserve os motivos de falha e o valor original.

In [0]:
# C07 — Regras e quarentena
regras = {
    "pedido_obrigatorio": F.coalesce(F.length(F.trim("pedido_id")) > 0, F.lit(False)),
    "cliente_obrigatorio": F.coalesce(F.length(F.trim("cliente_id")) > 0, F.lit(False)),
    "valor_positivo": F.coalesce(F.col("valor") > 0, F.lit(False)),
    "status_conhecido": F.coalesce(F.col("status").isin("pago", "cancelado"), F.lit(False)),
}
avaliados = atuais
for nome, regra in regras.items():
    avaliados = avaliados.withColumn(nome, regra)
ok = F.lit(True)
for nome in regras:
    ok = ok & F.col(nome)
avaliados = avaliados.withColumn("valido", ok).withColumn(
    "motivos", F.concat_ws(", ", *[F.when(~F.col(nome), F.lit(nome)) for nome in regras])
)
validos = avaliados.filter("valido")
quarentena = avaliados.filter("NOT valido")
display(quarentena.select("pedido_id", "valor_raw", "cliente_id", "motivos").orderBy("pedido_id"))


pedido_id,valor_raw,cliente_id,motivos
P04,null,C04,valor_positivo
P05,-50.00,C05,valor_positivo
P06,abc,C06,valor_positivo
P08,300.00,null,cliente_obrigatorio


## 5. Todas as linhas têm um destino?
Antes do resultado, vamos conferir as contagens.

In [0]:
# C08 — Reconciliação
n_raw, n_eventos, n_atuais = raw.count(), eventos.count(), atuais.count()
n_validos, n_quarentena = validos.count(), quarentena.count()
print(f"Recebidos: {n_raw}; eventos distintos: {n_eventos}; pedidos atuais: {n_atuais}")
print(f"Válidos: {n_validos}; quarentena: {n_quarentena}")
assert n_atuais == n_validos + n_quarentena
assert validos.select("pedido_id").distinct().count() == n_validos


Recebidos: 12; eventos distintos: 11; pedidos atuais: 10
Válidos: 6; quarentena: 4


## 6. A meta continua atingida?
Qual será o total dos pedidos pagos elegíveis? Execute a próxima célula depois de ouvir uma hipótese.

In [0]:
# C09 — Revelação
pagos = validos.filter(F.col("status") == "pago")
elegivel = pagos.agg(F.sum("valor").alias("total")).first()["total"]
resumo = spark.createDataFrame(
    [(1, "Meta", META), (2, "Relatório inicial", inicial), (3, "Elegível", elegivel)],
    "ordem int, etapa string, valor decimal(12,2)"
)
display(resumo.orderBy("ordem"))
print(f"Variação elegível em relação à meta: {(elegivel/META-1)*100:+.0f}%")
print("Diferença entre relatório e total elegível:", reais(inicial-elegivel))


ordem,etapa,valor
1,Meta,1000.00
2,Relatório inicial,1370.00
3,Elegível,800.00


Variação elegível em relação à meta: -20%
Diferença entre relatório e total elegível: R$ 570,00


## 7. Discussão e próximo passo
- E se valor negativo representar estorno? A regra muda e o indicador precisa definir seu tratamento.
- E se o cliente for opcional? O contrato pode liberar P08, levando o total elegível a R$ 1.100,00.
- Se uma versão mais recente vier inválida, preserve-a na quarentena e investigue a origem.
- Um contrato de produção também precisa de unicidade de eventos, validação da versão, integridade referencial e frescor. A demo não cobre todas as dimensões de qualidade.
- Preserve a camada bruta, registre motivos, dono da regra e versão do contrato. Reprocesse após a correção.
- Este notebook repete ações para facilitar a explicação. Em volume, consolide métricas, avalie cache e evite `collect()` de dados completos.

## Fontes
- [Semântica de NULL](https://spark.apache.org/docs/3.5.6/sql-ref-null-semantics.html)
- [Conversões e ANSI](https://spark.apache.org/docs/3.5.6/sql-ref-ansi-compliance.html)
- [dropDuplicates](https://spark.apache.org/docs/3.5.6/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.dropDuplicates.html)
- [row_number](https://spark.apache.org/docs/3.5.6/api/python/reference/pyspark.sql/api/pyspark.sql.functions.row_number.html)

[Agnes Ruescas no LinkedIn](https://www.linkedin.com/in/agnesruescas/)